In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
!pip install numpy==1.23.5
!pip install gensim

  Using cached gensim-4.3.3-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (8.1 kB)
  Using cached scipy-1.13.1-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (60 kB)
Using cached gensim-4.3.3-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (26.7 MB)
Using cached scipy-1.13.1-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (38.6 MB)
  Attempting uninstall: scipy
    Found existing installation: scipy 1.15.2
    Uninstalling scipy-1.15.2:
      Successfully uninstalled scipy-1.15.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
scikit-image 0.25.2 requires numpy>=1.24, but you have numpy 1.23.5 which is incompatible.
imbalanced-learn 0.13.0 requires numpy<3,>=1.24.3, but you have numpy 1.23.5 which is incompatible.
albumentations 2.0.5 requires numpy>=1.24.4, but you have numpy 1.23.5 which is inco

In [2]:
#埋め込みの読み込み
import numpy as np
from gensim.models import KeyedVectors

def load_pretrained_embeddings(embedding_path, vocab_limit=None):
    embeddings = []
    token2id = {}
    id2token = {}

    # GoogleNews-vectors-negative300.bin を読み込む
    word_vectors = KeyedVectors.load_word2vec_format(embedding_path, binary=True)

    embedding_dim = word_vectors.vector_size
    embeddings.append(np.zeros(embedding_dim))  # <PAD>トークン用ゼロベクトル
    token2id['<PAD>'] = 0
    id2token[0] = '<PAD>'

    for idx, word in enumerate(word_vectors.index_to_key):
        if vocab_limit and (idx >= vocab_limit):
            break
        vector = word_vectors[word]

        token_id = len(embeddings)
        token2id[word] = token_id
        id2token[token_id] = word
        embeddings.append(vector)

    embedding_matrix = np.vstack(embeddings) #各単語の埋め込みを縦方向に結合(ベクトル→行列)
    return embedding_matrix, token2id, id2token

# 使い方
embedding_path = 'drive/MyDrive/NLP100/8/GoogleNews-vectors-negative300.bin'
embedding_matrix, token2id, id2token = load_pretrained_embeddings(embedding_path, vocab_limit=50000)


In [3]:
#データセットの読み込み
import csv
import torch

def load_sst_data(file_path, token2id):
    dataset = []
    with open(file_path, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f, delimiter='\t')
        for row in reader:
            text = row['sentence']
            label = int(row['label'])

            # 単語ごとにトークンIDに変換
            tokens = text.split()
            input_ids = [token2id[token] for token in tokens if token in token2id]

            # 全部消えたらスキップ
            if len(input_ids) == 0:
                continue

            item = {
                'text': text,
                'label': torch.tensor([float(label)]),
                'input_ids': torch.tensor(input_ids)
            }
            dataset.append(item)
    return dataset

# 使い方
train_file = 'drive/MyDrive/NLP100/8//SST-2/train.tsv'
dev_file = 'drive/MyDrive/NLP100/8//SST-2/dev.tsv'

train_data = load_sst_data(train_file, token2id)
dev_data = load_sst_data(dev_file, token2id)


In [4]:
import torch
import torch.nn as nn

class TextAverageMLP(nn.Module):
    def __init__(self, embedding_matrix, hidden_dim=128):
        super(TextAverageMLP, self).__init__()
        vocab_size, embedding_dim = embedding_matrix.shape

        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.embedding.weight.data.copy_(torch.from_numpy(embedding_matrix))
        self.embedding.weight.requires_grad = False  # ファインチューニングなし

        self.fc1 = nn.Linear(embedding_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, 1)

    def forward(self, input_ids):
        embedded = self.embedding(input_ids)

        # マスクして平均
        mask = (input_ids != 0).unsqueeze(-1)
        masked_embeddings = embedded * mask
        summed = masked_embeddings.sum(dim=1)
        lengths = mask.sum(dim=1)
        avg_embedded = summed / lengths.clamp(min=1)

        x = self.fc1(avg_embedded)
        x = self.relu(x)
        x = self.fc2(x)
        probs = torch.sigmoid(x)
        return probs


In [11]:
#学習(パディング処理の追加)
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# --- 1. データセットとデータローダー準備 ---
class SSTDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        return {
            'input_ids': item['input_ids'],
            'label': item['label']
        }

def collate_batch(batch):
    input_ids = [item['input_ids'] for item in batch]
    labels = torch.stack([item['label'] for item in batch])

    input_ids_padded = torch.nn.utils.rnn.pad_sequence(
        input_ids, batch_first=True, padding_value=0
    )

    return {
        'input_ids': input_ids_padded,
        'label': labels
    }

# --- 2. データローダー作成 ---
train_dataset = SSTDataset(train_data)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_batch)

# --- 3. モデル準備 ---
model = TextAverageMLP(embedding_matrix)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCELoss()

# --- 4. 学習ループ ---
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for batch_idx, batch in enumerate(train_loader):
        input_ids = batch['input_ids']  # (バッチサイズ, シーケンス長)
        labels = batch['label']         # (バッチサイズ, 1)

        preds = model(input_ids)        # 順伝播
        loss = criterion(preds, labels) # 損失計算

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        # 正解率計算
        predicted = (preds >= 0.5).float()  # 閾値0.5でポジ/ネガ分類
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    avg_loss = running_loss / len(train_loader)
    accuracy = correct / total

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}, Accuracy: {accuracy:.4f}")


Epoch [1/10], Loss: 0.4125, Accuracy: 0.8119
Epoch [2/10], Loss: 0.3817, Accuracy: 0.8273
Epoch [3/10], Loss: 0.3655, Accuracy: 0.8366
Epoch [4/10], Loss: 0.3491, Accuracy: 0.8463
Epoch [5/10], Loss: 0.3336, Accuracy: 0.8548
Epoch [6/10], Loss: 0.3176, Accuracy: 0.8640
Epoch [7/10], Loss: 0.3030, Accuracy: 0.8728
Epoch [8/10], Loss: 0.2895, Accuracy: 0.8797
Epoch [9/10], Loss: 0.2766, Accuracy: 0.8856
Epoch [10/10], Loss: 0.2651, Accuracy: 0.8916


In [12]:
#予測
# --- 1. dev用データローダー作成 ---
dev_dataset = SSTDataset(dev_data)
dev_loader = DataLoader(dev_dataset, batch_size=32, shuffle=False, collate_fn=collate_batch)

# --- 2. 評価モードにして、推論 ---
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for batch in dev_loader:
        input_ids = batch['input_ids']  # (バッチサイズ, シーケンス長)
        labels = batch['label']         # (バッチサイズ, 1)

        preds = model(input_ids)

        predicted = (preds >= 0.5).float()
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

# --- 3. 正解率出力 ---
accuracy = correct / total
print(f"Dev Accuracy: {accuracy:.4f}")


Dev Accuracy: 0.8005
